# Post-`create_dataset` cleanup

This notebook does the following for a subset folder produced by `mdc.create_dataset` (e.g. `medical_datasets/malaria/`):

1. **Join step** — reads `image_metadata.json` and splits `cases.csv` into two files: `cases_with_image.csv`  and `cases_without_image.csv`.

In [43]:
# ===================== CONFIG =====================

import json
import re
from pathlib import Path

import pandas as pd

# Root folder that create_dataset generated, e.g. medical_datasets/dengue2
SUBSET_DIR = Path("medical_datasets/zika")

CASES_CSV = SUBSET_DIR / "cases.csv"
IMAGE_METADATA_JSON = SUBSET_DIR / "image_metadata.json"

CASES_WITH_IMAGE_CSV = SUBSET_DIR / "cases_with_image.csv"
CASES_WITHOUT_IMAGE_CSV = SUBSET_DIR / "cases_without_image.csv"

print(f"Subset dir: {SUBSET_DIR.resolve()}")

Subset dir: /home/tokuden/VGU_WS26_ClinicalProject_BHTBH/Demos/medical_datasets/zika


Join `cases.csv` with `image_metadata.json`

`image_metadata.json` is a list of image records, each with a `case_id` and an `image_subtype`. We collect the set of `case_id`s that have at least one image of a valid subtype, then keep only those rows of `cases.csv`.

In [44]:
with open(IMAGE_METADATA_JSON, "r", encoding="utf-8") as f:
    image_records = [json.loads(line) for line in f if line.strip()]


print(f"Total image records: {len(image_records)}")

# Sanity check: what image_subtypes actually exist in this file?
subtype_counts = pd.Series([r.get("image_subtype") for r in image_records]).value_counts(dropna=False)
print("\nimage_subtype breakdown:")
print(subtype_counts)

Total image records: 4

image_subtype breakdown:
skin_photograph    2
ultrasound         1
ct                 1
Name: count, dtype: int64


In [45]:
valid_case_ids = {
    r["case_id"]
    for r in image_records
}

print(f"Unique case_ids with a valid image: {len(valid_case_ids)}")

Unique case_ids with a valid image: 2


In [46]:
cases_df = pd.read_csv(CASES_CSV)
print(f"Total rows in cases.csv: {len(cases_df)}")

assert "case_id" in cases_df.columns, "cases.csv must have a case_id column"

has_image_mask = cases_df["case_id"].isin(valid_case_ids)

cases_with_image_df = cases_df[has_image_mask].copy()
cases_without_image_df = cases_df[~has_image_mask].copy()

print(f"Rows WITH a matching usable image:    {len(cases_with_image_df)}")
print(f"Rows WITHOUT a matching usable image: {len(cases_without_image_df)}")

cases_with_image_df.to_csv(CASES_WITH_IMAGE_CSV, index=False)
cases_without_image_df.to_csv(CASES_WITHOUT_IMAGE_CSV, index=False)

print(f"Saved -> {CASES_WITH_IMAGE_CSV}")
print(f"Saved -> {CASES_WITHOUT_IMAGE_CSV}")

Total rows in cases.csv: 12
Rows WITH a matching usable image:    2
Rows WITHOUT a matching usable image: 10
Saved -> medical_datasets/zika/cases_with_image.csv
Saved -> medical_datasets/zika/cases_without_image.csv
